### Cell 1: rebuilding table from the raw files

In [7]:
import pandas as pd
import numpy as np

wdi = pd.read_csv("../data/raw/wdi_2019.csv")
meta = pd.read_csv("../data/raw/country_metadata.csv")

ssa = meta[meta["Region"] == "Sub-Saharan Africa"][["Country Code", "TableName", "IncomeGroup"]]
data = ssa.merge(wdi, on="Country Code", how="left")
data = data.rename(columns={"TableName": "country"})
print(data.shape)

(48, 11)


### Cell 2: droping the weak variable and incomplete countries

In [8]:
data = data.drop(columns=["female_secondary_enrollment"])

before = len(data)
dropped = data[data.isna().any(axis=1)]["country"].tolist()
data = data.dropna().copy()

print(f"Dropped {before - len(data)} countries: {dropped}")
print(data.shape)

Dropped 3 countries: ['Eritrea', 'Equatorial Guinea', 'South Sudan']
(45, 10)


### Cell 3: taking the log of GDP

In [9]:
data["log_gdp_per_capita"] = np.log(data["gdp_per_capita"])
data = data.drop(columns=["gdp_per_capita"])
data[["country", "life_expectancy", "log_gdp_per_capita"]].head()

,country,life_expectancy,log_gdp_per_capita
0,Angola,63.051,7.827188
1,Burundi,62.180,5.456648
2,Benin,59.888,7.030700
3,Burkina Faso,60.184,6.639684
4,Botswana,67.167,8.877952


### Cell 4: which country spends 22% of GDP on health?

In [10]:
data.sort_values("health_spending_pct_gdp", ascending=False)[["country", "health_spending_pct_gdp"]].head(3)

,country,health_spending_pct_gdp
22,Lesotho,21.923731
1,Burundi,9.059406
29,Namibia,8.858058


### Cell 5: how strongly do the variables move together?

In [11]:
features = ["log_gdp_per_capita", "health_spending_pct_gdp", "electricity_access",
            "basic_water", "basic_sanitation", "urban_population_pct"]
data[["life_expectancy"] + features].corr().round(2)

,life_expectancy,log_gdp_per_capita,health_spending_pct_gdp,electricity_access,basic_water,basic_sanitation,urban_population_pct
life_expectancy,1.00,0.47,-0.20,0.57,0.55,0.57,0.19
log_gdp_per_capita,0.47,1.00,-0.08,0.81,0.77,0.69,0.53
health_spending_pct_gdp,-0.20,-0.08,1.00,-0.12,0.09,0.08,-0.18
electricity_access,0.57,0.81,-0.12,1.00,0.79,0.72,0.51
basic_water,0.55,0.77,0.09,0.79,1.00,0.73,0.52
basic_sanitation,0.57,0.69,0.08,0.72,0.73,1.00,0.28
urban_population_pct,0.19,0.53,-0.18,0.51,0.52,0.28,1.00


### Cell 6: saving the model table

In [12]:
data.to_csv("../data/processed/model_table.csv", index=False)
print("Saved", data.shape)

Saved (45, 10)
